# AlbumIQ — Live project evaluation
This notebook reads the **same database and stored images as your website**. Re-run the cells after uploading photos or searching on the web to refresh the results. It never changes photos, stored matches, or the model threshold.

Select **backend/.venv/bin/python** as the kernel. Start your usual database first. Outputs appear beneath each cell; no output folder is created.

**Operational percentages** measure project activity. **Recognition accuracy** measures correctness and needs human-reviewed answers; returning a photo does not prove the person matches.


## 1. Connect to AlbumIQ

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

root = Path.cwd().resolve()
while not (root / 'backend/config/settings.py').exists():
    if root == root.parent:
        raise RuntimeError('Open the AlbumIQ project folder first.')
    root = root.parent
sys.path.insert(0, str(root / 'evaluation'))
# Reload helpers when rerunning after a code change.
import project_status
import importlib
importlib.reload(project_status)
project_status.connect()
print('Connected to the project configuration. Database access runs in a safe worker thread.')


## 2. Refresh current website activity
Leave `EVENT_CODE = None` for all events, or enter an event code shown in the table.

In [ ]:
EVENT_CODE = None  # e.g. 'WED-UXTI'
state = project_status.database_call(project_status.snapshot, EVENT_CODE)
print('Snapshot:', state['captured_at'])
print('Model:', state['model'], '| threshold:', state['threshold'])
print('Stored photos:', len(state['photos']), '| recorded searches:', len(state['searches']))
print('Active-model embeddings:', state['current_embeddings'])
display(state['events'])
display(state['metrics'].style.format({'Percentage (%)': '{:.2f}%'}, na_rep='No observations'))
print('These are operational rates, not verified recognition accuracy.')


## 3. Live percentage graphs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rates = state['metrics']
values = rates['Percentage (%)'].to_numpy()
axes[0].barh(rates.Metric, np.nan_to_num(values), color='#8f503f')
for i, value in enumerate(values):
    axes[0].text((value if np.isfinite(value) else 0)+1, i,
                 f'{value:.2f}%' if np.isfinite(value) else 'No observations', va='center')
axes[0].set(xlim=(0, 125), xlabel='Percentage (%)', title='Current operational rates — not accuracy')
if not state['events'].empty:
    state['events'].set_index('Event')[['Photos', 'Searchable']].plot.bar(ax=axes[1])
axes[1].set(title='Uploaded vs searchable photos by event', ylabel='Photo count')
plt.tight_layout()
plt.show()


## 4. Re-run stored website selfies against current albums
This reuses your website's actual stored selfies, preprocessing, face embeddings, threshold and event filtering. It does **not** save a new search or update old counts. Stored counts may differ because you uploaded more photos or changed the model since the original search.

`GuestSearch` does not store returned photo IDs, success/error flags, or historical timing. This cell measures a **current replay**, not historical accuracy or historical response time. First model load is included in replay timings.


In [ ]:
SEARCH_LIMIT = 3  # latest searches; increase to replay more website activity
queries, matches, galleries = project_status.database_call(project_status.replay, EVENT_CODE, SEARCH_LIMIT)
display(queries)
display(matches)
if queries.empty:
    print('No stored searches for this selection. Upload a selfie on the website, then rerun this cell.')
else:
    valid_queries = queries[queries.status == 'OK']
    print(f'Replay processing coverage: {100 * len(valid_queries) / len(queries):.2f}%')
    print('Returned photos are predictions; they are not automatically correct labels.')


## 5. View real images and identify the correct photos
Choose a `search_id` from step 4. The reference selfie and album thumbnails show below with photo IDs. Use `PAGE` to review **all pages** before labeling the complete relevant-photo set. Seeing only returned results cannot measure missed matches.


In [ ]:
SEARCH_ID = queries.iloc[0]['search_id'] if not queries.empty else None
PAGE = 1
if SEARCH_ID:
    images, total = project_status.database_call(project_status.review_page, SEARCH_ID, PAGE)
    fig, axes = plt.subplots(int(np.ceil(len(images)/4)), 4, figsize=(14, 3.5*np.ceil(len(images)/4)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (label, image) in zip(axes.flat, images):
        if image is not None: ax.imshow(image)
        else: ax.text(.5,.5,'Image unavailable',ha='center')
        ax.set_title(label)
    plt.tight_layout()
    plt.show()
    print(f'Album: {total} photos | Page {PAGE} of {max(1,int(np.ceil(total/12)))}')
else:
    print('No website selfie is available to review yet.')


## 6. Actual recognition accuracy from reviewed website activity
After inspecting every album page for a selfie, enter **all photo IDs containing that person**. This is the correct answer, independent of what the model returned. An empty list means you confirmed the person is absent from the whole album.

Example structure only: `GROUND_TRUTH = {'paste-search-uuid-here': [12, 18, 25]}`. Use your real IDs from steps 4–5. No files or database edits are necessary.


In [ ]:
GROUND_TRUTH = {
    # 'paste-real-search-id-here': [photo_id_1, photo_id_2],
}
reviewed = project_status.reviewed_accuracy(queries, matches, galleries, GROUND_TRUTH)
if reviewed.empty:
    display(Markdown('**Accuracy awaiting review.** Live counts and graphs above already use your web data. Add the correct photo IDs here to measure whether those predictions are right.'))
else:
    display(reviewed.style.format({c: '{:.2f}%' for c in reviewed.columns if c.endswith('(%)')}, na_rep='Undefined'))
    percentage_columns = [c for c in reviewed.columns if c.endswith('(%)')]
    mean_scores = reviewed[percentage_columns].mean()
    display(mean_scores.rename('Mean across reviewed searches (%)').to_frame().round(2))
    print('Photo decision accuracy includes correct rejections and may be inflated by many irrelevant photos. Read precision and recall alongside it.')
    print('Failed replay queries receive zero accuracy; undefined metrics are not replaced with invented scores.')


## 7. Verified accuracy graphs

In [ ]:
if reviewed.empty:
    print('Complete step 6 to graph verified accuracy. Predictions alone cannot establish accuracy.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    scores = reviewed[[c for c in reviewed.columns if c.endswith('(%)')]].mean()
    scores.plot.bar(ax=axes[0], color='#8f503f')
    axes[0].set(ylim=(0,110), ylabel='Percentage (%)', title=f'Mean metrics — {len(reviewed)} reviewed searches')
    axes[0].tick_params(axis='x', rotation=35)
    for i, value in enumerate(scores):
        if np.isfinite(value): axes[0].text(i,value+1,f'{value:.2f}%',ha='center')
    valid = reviewed[reviewed['Query processed']]
    counts = valid[['TN','FP','FN','TP']].sum()
    matrix = np.array([[counts.TN, counts.FP], [counts.FN, counts.TP]])
    axes[1].imshow(matrix, cmap='Blues')
    for i in range(2):
        for j in range(2): axes[1].text(j,i,str(matrix[i,j]),ha='center',va='center')
    axes[1].set(xticks=[0,1],yticks=[0,1],xticklabels=['Absent','Present'],yticklabels=['Absent','Present'],
                xlabel='Predicted person presence',ylabel='Reviewed person presence',title='Confusion counts — processed reviewed queries')
    plt.tight_layout()
    plt.show()


### Refresh after using the website
Re-run **steps 2–4** after uploads or searches. Review new queries and rerun steps 5–7 for verified scores. If the album changes, recheck its ground truth before reusing labels.

These scores describe reviewed current-project searches, not an independent research test. Saved search counts are not accuracy, and no success labels are inferred from filenames or predictions. Failed/abandoned searches cannot be distinguished retrospectively because the website does not log their outcome separately.
